In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


In [2]:
torch.manual_seed(42)

In [3]:
df = pd.read_csv('fmnist_small.csv')

In [4]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [5]:
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
X_train = X_train/255.0
X_test = X_test/255.0

In [8]:
# Create custom dataclass
class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32).reshape(-1, 1, 28, 28)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [9]:
train_dataset = CustomDataset(X_train, y_train)

In [10]:
test_dataset = CustomDataset(X_test, y_test)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [12]:
class MyNN(nn.Module):
    def __init__(self, input_channel):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(input_channel, 32, kernel_size=3, padding='same'),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding='same'),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(64, 10)
        )

    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x

In [13]:
learning_rate = 0.01
epochs = 50

In [14]:
model = MyNN(1)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [15]:
# training loop
for epoch in range(epochs):
    total_epoch_loss = 0

    for batch_features, batch_labels in train_loader:
        outputs = model(batch_features)

        loss = criterion(outputs, batch_labels)

        optimizer.zero_grad()
        loss.backward()

        optimizer.step()

        total_epoch_loss = total_epoch_loss + loss.item()

        avg_loss = total_epoch_loss/len(train_loader)
    print(f"Epoch: {epoch+1}, Loss: {avg_loss}")

/home/roben/Codes/PracticalDeepLearning/practicaldeeplearning/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch: 1, Loss: 1.6531695520877838
Epoch: 2, Loss: 1.0102613572279613
Epoch: 3, Loss: 0.8317067813873291
Epoch: 4, Loss: 0.7181850997606913
Epoch: 5, Loss: 0.6698802296320597
Epoch: 6, Loss: 0.6156211525201798
Epoch: 7, Loss: 0.5900215752919515
Epoch: 8, Loss: 0.5462773052851359
Epoch: 9, Loss: 0.5125708550214767
Epoch: 10, Loss: 0.49170961836973825
Epoch: 11, Loss: 0.4606803384423256
Epoch: 12, Loss: 0.4461357593536377
Epoch: 13, Loss: 0.42076751619577407
Epoch: 14, Loss: 0.4037558116515477
Epoch: 15, Loss: 0.39081905235846837
Epoch: 16, Loss: 0.36648299361268677
Epoch: 17, Loss: 0.3471687329808871
Epoch: 18, Loss: 0.347152314633131
Epoch: 19, Loss: 0.31385862454771996
Epoch: 20, Loss: 0.30802148843804994
Epoch: 21, Loss: 0.2870445705453555
Epoch: 22, Loss: 0.27680998692909875
Epoch: 23, Loss: 0.2691879831751188
Epoch: 24, Loss: 0.25340177938342096
Epoch: 25, Loss: 0.2414998176942269
Epoch: 26, Loss: 0.225927835876743
Epoch: 27, Loss: 0.21720059007406234
Epoch: 28, Loss: 0.21600874679

In [16]:
# set model to eval
model.eval()

MyNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [21]:
# evaluation code
total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_label in test_loader:
        outputs = model(batch_features)

        _, predicted = torch.max(outputs, 1)

        total += batch_label.shape[0]

        correct += (predicted == batch_label).sum().item()

print(correct/total)

0.8658333333333333


In [22]:
# evaluation on training data
total = 0
correct = 0

with torch.no_grad():

  for batch_features, batch_labels in train_loader:

    # move data to gpu

    outputs = model(batch_features)

    _, predicted = torch.max(outputs, 1)

    total = total + batch_labels.shape[0]

    correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.99625
